In [27]:
import glob
import os
import random
import numpy as np
import csv
import json

In [28]:
class MajorityClassifier:
    def fit(self, X, y):
        counts = {}
        for label in y:
            counts[label] = counts.get(label, 0) + 1
        self.most_frequent = max(counts, key=counts.get)
        return self
 
    def predict(self, X):
        return [self.most_frequent for _ in X]   

In [29]:
def train_test_split(X, y, test_size=0.2, seed=42):
    idx = list(range(len(y)))
    random.Random(seed).shuffle(idx)
    n_test = int(len(y) * test_size)
    test, train = idx[:n_test], idx[n_test:]
    return ([X[i] for i in train], [X[i] for i in test],
            [y[i] for i in train], [y[i] for i in test])

In [30]:
def get_accuracy(y_true, y_pred):
    correct = sum(1 for t, p in zip(y_true, y_pred) if t == p)
    return correct / len(y_true)

In [31]:
DATA_ROOT = "/path/to/autoPET"
 
# Build labels, AI helped with this snippet
X, y = [], []
for path in sorted(glob.glob(os.path.join(DATA_ROOT, "labelsTr", "*.nii.gz"))):
    mask = nib.load(path).dataobj
    X.append(os.path.basename(path))
    y.append(1 if mask[...].any() else 0)
 
print(f"Scans: {len(y)}  |  tumor: {sum(y)}  |  no tumor: {len(y) - sum(y)}") # AI for print

Scans: 0  |  tumor: 0  |  no tumor: 0


In [32]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
#clf = MajorityClassifier().fit(X_train, y_train)
#print(f"Most frequent class: {clf.most_frequent}") # AI for print
#y_pred = clf.predict(X_test)
#print(f"Baseline Accuracy: {get_accuracy(y_test, y_pred):.4f}") # AI for print

In [33]:
# Load FDG labels (AI helped with this)
labels = {}
with open("data/fdg_metadata.csv") as f:
    for row in csv.DictReader(f):
        if row["Modality"] == "PT":
            patient = row["Subject ID"].replace("PETCT_", "")
            study = row["File Location"].split("/")[-2]
            name = f"fdg_{patient}_{study}"
            labels[name] = 0 if row["diagnosis"] == "NEGATIVE" else 1

# Load the first training/validation split
with open("data/splits_final.json") as f:
    split = json.load(f)[0]

X_train = [x for x in split["train"] if x in labels]
X_test = [x for x in split["val"] if x in labels]

y_train = [labels[x] for x in X_train]
y_test = [labels[x] for x in X_test]

# Test existing classifier
model = MajorityClassifier().fit(X_train, y_train)
predictions = model.predict(X_test)

print("Training scans:", len(X_train))
print("Testing scans:", len(X_test))
print("Accuracy:", get_accuracy(y_test, predictions))

Training scans: 822
Testing scans: 192
Accuracy: 0.4375


In [ ]:
#Binary classification components. no encoder or pretrained weights included

import torch
from torch import nn


class BinaryClassificationHead(nn.Module):
    #Map [B, C, D, H, W] features (or pooled [B, C]) to [B, 1] logits.

    def __init__(self, in_features: int):
        super().__init__()
        if in_features <= 0:
            raise ValueError("in_features must be positive")
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Linear(in_features, 1)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        if not isinstance(features, torch.Tensor):
            raise TypeError("Expected a feature tensor; adapt the encoder output explicitly")
        if features.ndim not in (2, 5):
            raise ValueError("Expected [B, C] or [B, C, D, H, W] features")
        if features.shape[1] != self.fc.in_features:
            raise ValueError("Feature channels must match the configured in_features")
        if features.ndim == 5:
            features = self.pool(features).flatten(1)
        return self.fc(features)

In [ ]:
from copy import deepcopy


def train_classifier(X_train, y_train, X_val, y_val, epochs=100,
                     hidden_size=None, weight_decay=0.0, verbose=True):
    features = torch.as_tensor(X_train, dtype=torch.float32)
    labels = torch.as_tensor(y_train, dtype=torch.float32).reshape(-1, 1)
    val_features = torch.as_tensor(X_val, dtype=torch.float32)
    val_labels = torch.as_tensor(y_val, dtype=torch.float32).reshape(-1, 1)
    if hidden_size is None:
        model = BinaryClassificationHead(in_features=features.shape[1])
    else:
        model = nn.Sequential(nn.Linear(features.shape[1], hidden_size),
                              nn.ReLU(), nn.Linear(hidden_size, 1))

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=weight_decay)
    training_losses = []
    validation_losses = []
    best_validation_loss = float("inf")
    best_weights = None
    best_epoch = 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        logits = model(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        training_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_logits = model(val_features)
            val_loss = criterion(val_logits, val_labels)
        validation_losses.append(val_loss.item())

        if val_loss.item() < best_validation_loss:
            best_validation_loss = val_loss.item()
            best_weights = deepcopy(model.state_dict())
            best_epoch = epoch + 1

        if verbose and (epoch == 0 or (epoch + 1) % 10 == 0):
            print(f"Epoch {epoch + 1}/{epochs}, Train loss: {loss.item():.4f}, Val loss: {val_loss.item():.4f}")

    model.load_state_dict(best_weights)
    model.eval()
    if verbose:
        print(f"Selected epoch: {best_epoch}, Validation loss: {best_validation_loss:.4f}")
    return model, training_losses, validation_losses, best_epoch


In [ ]:
# Load the saved features.
import numpy as np

with np.load("features/encoder_features.npz") as data:
    encoder_features = data["features"]
    tumor_labels = data["lesion"]
    feature_split = data["split"]

train_features = encoder_features[feature_split == "train"]
train_labels = tumor_labels[feature_split == "train"]
val_features = encoder_features[feature_split == "val"]
val_labels = tumor_labels[feature_split == "val"]
test_features = encoder_features[feature_split == "test"]
test_labels = tumor_labels[feature_split == "test"]

print("Training features:", train_features.shape)
print("Validation features:", val_features.shape)
print("Test features:", test_features.shape)

print(f"Training tumor fraction: {train_labels.mean():.2%}")
print("Training feature standard deviations (min / median / max):",
      np.quantile(train_features.std(axis=0), [0, 0.5, 1]))


Training features: (980, 640)
Validation features: (315, 640)
Test features: (316, 640)
Training tumor fraction: 64.80%
Training feature standard deviations (min / median / max): [0.00175164 0.00660338 0.02567509]


In [ ]:
# Compare raw and standardized features.
torch.manual_seed(42)
original_classifier, _, original_validation_losses, original_best_epoch = train_classifier(
    train_features, train_labels, val_features, val_labels, epochs=100
)

# Use training statistics for every split.
feature_mean = train_features.mean(axis=0, keepdims=True)
feature_std = train_features.std(axis=0, keepdims=True)
feature_std[feature_std < 1e-6] = 1
scaled_train_features = (train_features - feature_mean) / feature_std
scaled_val_features = (val_features - feature_mean) / feature_std

torch.manual_seed(42)
feature_classifier, training_losses, validation_losses, best_epoch = train_classifier(
    scaled_train_features, train_labels, scaled_val_features, val_labels, epochs=100
)
print(f"First scaled validation loss: {validation_losses[0]:.4f}")
print(f"Best scaled validation loss: {min(validation_losses):.4f}")
print(f"Last scaled validation loss: {validation_losses[-1]:.4f}")


Epoch 1/100, Train loss: 0.8125, Val loss: 0.7458
Epoch 10/100, Train loss: 0.6819, Val loss: 0.6508
Epoch 20/100, Train loss: 0.6555, Val loss: 0.6491
Epoch 30/100, Train loss: 0.6497, Val loss: 0.6320
Epoch 40/100, Train loss: 0.6487, Val loss: 0.6349
Epoch 50/100, Train loss: 0.6485, Val loss: 0.6326
Epoch 60/100, Train loss: 0.6484, Val loss: 0.6335
Epoch 70/100, Train loss: 0.6483, Val loss: 0.6325
Epoch 80/100, Train loss: 0.6481, Val loss: 0.6331
Epoch 90/100, Train loss: 0.6479, Val loss: 0.6324
Epoch 100/100, Train loss: 0.6478, Val loss: 0.6325
Selected epoch: 35, Validation loss: 0.6320
Epoch 1/100, Train loss: 0.6817, Val loss: 0.6601
Epoch 10/100, Train loss: 0.6232, Val loss: 0.6258
Epoch 20/100, Train loss: 0.5962, Val loss: 0.6167
Epoch 30/100, Train loss: 0.5783, Val loss: 0.6054
Epoch 40/100, Train loss: 0.5637, Val loss: 0.5978
Epoch 50/100, Train loss: 0.5514, Val loss: 0.5952
Epoch 60/100, Train loss: 0.5405, Val loss: 0.5933
Epoch 70/100, Train loss: 0.5306, Val l

In [ ]:
# Confusion matrix: actual rows, predicted columns; labels 0, 1.
def classification_metrics(actual, predicted):
    actual = np.asarray(actual).reshape(-1)
    predicted = np.asarray(predicted).reshape(-1)
    assert actual.shape == predicted.shape
    tn = int(((actual == 0) & (predicted == 0)).sum())
    fp = int(((actual == 0) & (predicted == 1)).sum())
    fn = int(((actual == 1) & (predicted == 0)).sum())
    tp = int(((actual == 1) & (predicted == 1)).sum())
    assert tn + fp > 0 and tp + fn > 0, "Both classes are needed for balanced accuracy"
    accuracy = (tn + tp) / len(actual)
    balanced_accuracy = (tn / (tn + fp) + tp / (tp + fn)) / 2
    return accuracy, balanced_accuracy, np.array([[tn, fp], [fn, tp]])


def compare_models(features, labels, split_name):
    feature_classifier.eval()
    with torch.no_grad():
        scaled_features = (features - feature_mean) / feature_std
        inputs = torch.as_tensor(scaled_features, dtype=torch.float32)
        predictions = (feature_classifier(inputs) >= 0).int().flatten().numpy()
    baseline_predictions = feature_baseline.predict(features)
    results = {}
    for name, predicted in (("Classification head", predictions),
                            ("Majority baseline", baseline_predictions)):
        accuracy, balanced_accuracy, matrix = classification_metrics(labels, predicted)
        results[name] = {"accuracy": accuracy, "balanced_accuracy": balanced_accuracy,
                         "confusion_matrix": matrix}
        print(f"{split_name} - {name}")
        print(f"Accuracy: {accuracy:.2%}, Balanced accuracy: {balanced_accuracy:.2%}")
        print("Confusion matrix (actual rows / predicted columns, order 0, 1):")
        print(matrix)
        tn, fp = matrix[0]
        fn, tp = matrix[1]
        print(f"Sensitivity: {tp / (tp + fn):.2%}, Specificity: {tn / (tn + fp):.2%}")
    return results


feature_baseline = MajorityClassifier().fit(train_features, train_labels)
print("Training majority class:", feature_baseline.most_frequent)
validation_results = compare_models(val_features, val_labels, "Validation")

with torch.no_grad():
    raw_val_inputs = torch.as_tensor(val_features, dtype=torch.float32)
    original_probabilities = original_classifier(raw_val_inputs).sigmoid().flatten().numpy()
    scaled_probabilities = feature_classifier(
        torch.as_tensor(scaled_val_features, dtype=torch.float32)
    ).sigmoid().flatten().numpy()
original_accuracy, original_balanced_accuracy, original_matrix = classification_metrics(
    val_labels, original_probabilities >= 0.5
)
print(f"Original validation probability range: {original_probabilities.min():.4f} to {original_probabilities.max():.4f}")
print(f"Scaled validation probability range: {scaled_probabilities.min():.4f} to {scaled_probabilities.max():.4f}")
print(f"Original accuracy: {original_accuracy:.2%}, Balanced accuracy: {original_balanced_accuracy:.2%}")
print("Original confusion matrix:")
print(original_matrix)


Training majority class: 1
Validation - Classification head
Accuracy: 67.94%, Balanced accuracy: 70.94%
Confusion matrix (actual rows / predicted columns, order 0, 1):
[[ 82  21]
 [ 80 132]]
Sensitivity: 62.26%, Specificity: 79.61%
Validation - Majority baseline
Accuracy: 67.30%, Balanced accuracy: 50.00%
Confusion matrix (actual rows / predicted columns, order 0, 1):
[[  0 103]
 [  0 212]]
Sensitivity: 100.00%, Specificity: 0.00%
Original validation probability range: 0.6718 to 0.6768
Scaled validation probability range: 0.0484 to 0.9426
Original accuracy: 67.30%, Balanced accuracy: 50.00%
Original confusion matrix:
[[  0 103]
 [  0 212]]


In [ ]:
# Test evaluation stays paused while tuning on validation data.


### Feature scaling

The raw features change very little between scans (median training standard deviation: 0.0066). The unscaled head gave probabilities of 0.6718–0.6768, so every prediction was positive.

Following HW6, each feature is scaled using its training mean and standard deviation. Both runs use seed 42, Adam at 0.001, and 100 epochs. Weights are selected by validation loss.

| Validation metric | Unscaled head | Scaled head (v1) |
|---|---:|---:|
| Accuracy | 67.30% | 67.94% |
| Balanced accuracy | 50.00% | 70.94% |
| Sensitivity | 100.00% | 62.26% |
| Specificity | 0.00% | 79.61% |
| BCE loss | 0.6320 | 0.5898 |

V1 selected epoch 100. Its confusion matrix is `[[82, 21], [80, 132]]` (actual rows, predicted columns; labels 0, 1). Validation loss was still falling at the end of the run.

### Earlier test result

Before scaling, the head and majority baseline both had 61.08% test accuracy and 50.00% balanced accuracy. Neither v1 nor v2 has been evaluated on the test set. Test evaluation is paused during tuning. Since the earlier test result is already known, a new holdout would give a stronger final check.

### Encoder checkpoint

The training script starts from Dataset619's `fold_all/checkpoint_final.pth`. The original extractor could choose final, best, or latest fold-0 weights, but the saved archive does not say which it used. Caleb/Vance still needs to confirm the checkpoint and that test patients were excluded from encoder training and selection. The updated extractor now requires a checkpoint name and hash.


### More validation experiments

V1 weights and scaling statistics are saved in `checkpoints/standardized_v1.pt`. The notebook snapshot is `checkpoints/classifier_standardized_v1.ipynb`.

Try up to 500 epochs with linear heads (weight decay 0, 0.01, 0.1) and one 16-neuron ReLU hidden layer (weight decay 0.01, 0.1). Keep the best validation-loss epoch from each run, then select the model with the lowest validation loss, including v1.

For that model, compare thresholds from 0.40 to 0.70 in steps of 0.05. Choose the best validation balanced accuracy; break ties by accuracy, then closeness to 0.5. The test set is not used. These are validation-selected scores, not final test results.


In [ ]:
# Load v1 without overwriting it.
from pathlib import Path
import hashlib

checkpoint_dir = Path("checkpoints")
checkpoint_dir.mkdir(exist_ok=True)
v1_path = checkpoint_dir / "standardized_v1.pt"
if not v1_path.exists():
    raise FileNotFoundError("Restore standardized_v1.pt from your saved checkpoint folder first")
v1_checkpoint = torch.load(v1_path, map_location="cpu", weights_only=True)
feature_hash = hashlib.sha256(Path("features/encoder_features.npz").read_bytes()).hexdigest()
assert feature_hash == v1_checkpoint["features_sha256"]
v1_classifier = BinaryClassificationHead(v1_checkpoint["in_features"])
v1_classifier.load_state_dict(v1_checkpoint["model_state_dict"])
v1_classifier.eval()
print("Preserved version 1:", v1_path)
print("Validation metrics:", v1_checkpoint["validation_metrics"])


Preserved version 1: checkpoints\standardized_v1.pt
Validation metrics: {'accuracy': 0.6793650793650794, 'balanced_accuracy': 0.7093790071441656, 'confusion_matrix': [[82, 21], [80, 132]], 'bce_loss': 0.5898410081863403}


In [ ]:
# Compare five settings against v1.
experiments = [{"name": "Saved linear v1", "model": v1_classifier,
                "hidden_size": None, "weight_decay": 0.0, "epochs": 100,
                "best_epoch": v1_checkpoint["selected_epoch"],
                "val_loss": v1_checkpoint["validation_metrics"]["bce_loss"]}]
settings = [("Linear, longer", None, 0.0),
            ("Linear, decay 0.01", None, 0.01),
            ("Linear, decay 0.1", None, 0.1),
            ("16 hidden, decay 0.01", 16, 0.01),
            ("16 hidden, decay 0.1", 16, 0.1)]

for name, hidden_size, decay in settings:
    torch.manual_seed(42)
    candidate, train_history, val_history, selected_epoch = train_classifier(
        scaled_train_features, train_labels, scaled_val_features, val_labels,
        epochs=500, hidden_size=hidden_size, weight_decay=decay, verbose=False
    )
    experiments.append({"name": name, "model": candidate, "hidden_size": hidden_size,
                        "weight_decay": decay, "epochs": 500,
                        "best_epoch": selected_epoch, "val_loss": min(val_history)})

print("Model comparison (best validation BCE loss):")
for experiment in experiments:
    print(f"{experiment['name']}: epoch {experiment['best_epoch']}, loss {experiment['val_loss']:.4f}")
selected = min(experiments, key=lambda experiment: experiment["val_loss"])
improved_classifier = selected["model"]
improved_classifier.eval()
print("Selected:", selected["name"])


Model comparison (best validation BCE loss):
Saved linear v1: epoch 100, loss 0.5898
Linear, longer: epoch 231, loss 0.5862
Linear, decay 0.01: epoch 500, loss 0.5751
Linear, decay 0.1: epoch 500, loss 0.5507
16 hidden, decay 0.01: epoch 66, loss 0.5230
16 hidden, decay 0.1: epoch 355, loss 0.5257
Selected: 16 hidden, decay 0.01


In [ ]:
# Choose the threshold using validation data.
with torch.no_grad():
    val_inputs = torch.as_tensor(scaled_val_features, dtype=torch.float32)
    improved_probabilities = improved_classifier(val_inputs).sigmoid().flatten().numpy()
threshold_results = []
for threshold in (0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70):
    accuracy, balanced_accuracy, matrix = classification_metrics(
        val_labels, improved_probabilities >= threshold
    )
    threshold_results.append({"threshold": threshold, "accuracy": accuracy,
                              "balanced_accuracy": balanced_accuracy,
                              "confusion_matrix": matrix})
    print(f"Threshold {threshold:.2f}: accuracy {accuracy:.2%}, balanced accuracy {balanced_accuracy:.2%}")

improved_results = max(threshold_results, key=lambda result: (
    result["balanced_accuracy"], result["accuracy"], -abs(result["threshold"] - 0.5)
))
improved_threshold = improved_results["threshold"]
tn, fp = improved_results["confusion_matrix"][0]
fn, tp = improved_results["confusion_matrix"][1]
print(f"Selected threshold: {improved_threshold:.2f}")
print(f"Validation accuracy: {improved_results['accuracy']:.2%}")
print(f"Validation balanced accuracy: {improved_results['balanced_accuracy']:.2%}")
print(f"Tumor sensitivity: {tp / (tp + fn):.2%}, Specificity: {tn / (tn + fp):.2%}")
print("Confusion matrix (actual rows, predicted columns; label order 0, 1):")
print(improved_results["confusion_matrix"])


Threshold 0.40: accuracy 73.33%, balanced accuracy 64.71%
Threshold 0.45: accuracy 72.70%, balanced accuracy 66.49%
Threshold 0.50: accuracy 73.65%, balanced accuracy 69.44%
Threshold 0.55: accuracy 72.06%, balanced accuracy 71.51%
Threshold 0.60: accuracy 68.25%, balanced accuracy 70.42%
Threshold 0.65: accuracy 66.03%, balanced accuracy 69.77%
Threshold 0.70: accuracy 66.03%, balanced accuracy 71.27%
Selected threshold: 0.55
Validation accuracy: 72.06%
Validation balanced accuracy: 71.51%
Tumor sensitivity: 73.11%, Specificity: 69.90%
Confusion matrix (actual rows, predicted columns; label order 0, 1):
[[ 72  31]
 [ 57 155]]


In [ ]:
# Save v2 separately and check that it reloads.
v2_path = checkpoint_dir / "improved_v2.pt"
if v2_path.exists():
    print("Version 2 already exists; preserving it. Use a new filename for another run.")
else:
    torch.save({"model_state_dict": improved_classifier.state_dict(),
                "feature_mean": torch.tensor(feature_mean), "feature_std": torch.tensor(feature_std),
                "in_features": int(train_features.shape[1]), "threshold": improved_threshold,
                "selected_epoch": selected["best_epoch"], "seed": 42,
                "config": {"epochs": selected["epochs"], "lr": 0.001, "optimizer": "Adam",
                           "hidden_size": selected["hidden_size"], "weight_decay": selected["weight_decay"]},
                "validation_metrics": {"accuracy": improved_results["accuracy"],
                    "balanced_accuracy": improved_results["balanced_accuracy"],
                    "confusion_matrix": improved_results["confusion_matrix"].tolist(),
                    "bce_loss": selected["val_loss"]},
                "features_sha256": feature_hash, "torch_version": str(torch.__version__),
                "encoder_checkpoint": "unconfirmed"}, v2_path)
    print("Saved:", v2_path)

checkpoint = torch.load(v2_path, map_location="cpu", weights_only=True)
hidden_size = checkpoint["config"].get("hidden_size")
if hidden_size is None:
    restored_classifier = BinaryClassificationHead(checkpoint["in_features"])
else:
    restored_classifier = nn.Sequential(nn.Linear(checkpoint["in_features"], hidden_size),
                                        nn.ReLU(), nn.Linear(hidden_size, 1))
restored_classifier.load_state_dict(checkpoint["model_state_dict"])
restored_classifier.eval()
with torch.no_grad():
    restored_inputs = (torch.as_tensor(val_features, dtype=torch.float32)
                       - checkpoint["feature_mean"]) / checkpoint["feature_std"]
    restored_probabilities = restored_classifier(restored_inputs).sigmoid().flatten()
    torch.testing.assert_close(restored_probabilities, torch.as_tensor(improved_probabilities))
print("Checkpoint reload reproduces validation probabilities.")


Saved: checkpoints\improved_v2.pt
Checkpoint reload reproduces validation probabilities.


### V2 results

V2 uses 640 -> 16 -> 1 layers with ReLU, Adam at 0.001, and weight decay 0.01. Epoch 66 had the lowest validation loss. The selected probability threshold is 0.55.

| Validation metric | V1 | V2 |
|---|---:|---:|
| Accuracy | 67.94% | 72.06% |
| Balanced accuracy | 70.94% | 71.51% |
| Sensitivity | 62.26% | 73.11% |
| Specificity | 79.61% | 69.90% |
| BCE loss | 0.5898 | 0.5230 |

V2's confusion matrix is `[[72, 31], [57, 155]]`. It catches 23 more tumors than v1 but adds 10 false positives. At threshold 0.5, accuracy was 73.65% and balanced accuracy was 69.44%; 0.55 gives a more even result across classes.

Weights, scaling statistics, thresholds, settings, feature-file hashes, and validation results are saved in `checkpoints/standardized_v1.pt` and `checkpoints/improved_v2.pt`. Reloaded predictions matched the originals. The files do not include Adam state for resuming training.

To restore v1, change the restore cell's path to `standardized_v1.pt`. Always use the checkpoint's mean, standard deviation, and threshold for predictions. Both versions still need final testing, and the encoder checkpoint remains unconfirmed.
